# Causal ranking feature dataset

BT4222 Group 6 · feature engineering. Run the three `3_Representations` notebooks (3_01, 3_02, 3_03) first.

One shared feature API supports complete logged slates and unlabelled retrieved Top-M candidates.
Historical state always consumes the **complete earlier exposure stream**, even when emitted training
pairs are sampled. Validation/test slates stay complete. This notebook defaults to a chronological smoke
build; set `MAX_TIMESTAMPS=None` for all agreed data once the smoke audit passes.

Run top to bottom using the same Python environment as the other notebooks. All project code and settings are included below; no local Python module or settings file is imported. Upstream Parquet/NumPy artifacts remain the inputs shared between notebooks.

`REBUILD=False` reuses verified saved outputs; set it to `True` after changing this notebook's calculations/settings. Restarting VS Code clears variables, not saved datasets.

In [ ]:
# All project settings and implementation are in this notebook.
# Standard third-party packages come from requirements.txt.
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import google.colab
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

# Edit DATA_ROOT to point to another project/data folder, if necessary.
DATA_ROOT = None
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/BT4222")]
if DATA_ROOT is not None:
    ROOT = Path(DATA_ROOT).expanduser().resolve()
elif os.environ.get("NEWSREC_ROOT"):
    ROOT = Path(os.environ["NEWSREC_ROOT"]).expanduser().resolve()
else:
    ROOT = next(
        (
            p
            for p in candidates
            if (p / "1_Data_Preparation/processed/news_catalogue.parquet").is_file()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Run data preparation first, or set DATA_ROOT above.")
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data and output root:", ROOT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
from collections import Counter
import polars as pl
import sqlite3
from collections import Counter, deque
from dataclasses import dataclass, field
import heapq
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

### Artifact checks and reporting

In [ ]:
NOTEBOOK_PATHS = (
    "3_Representations/3_01_news_embeddings.ipynb",
    "3_Representations/3_02_entity_embeddings.ipynb",
    "3_Representations/3_03_user_profiles.ipynb",
    "4_Feature_Engineering/4_01_build_ranking_features.ipynb",
)

In [ ]:
SPLITS = ("train", "validation", "test")

In [ ]:
VERSION = "newsrec-v1"

In [ ]:
@dataclass(frozen=True)
class Settings:
    seed: int = 4222
    text_dim: int = 64
    max_features: int = 50000
    min_df: int = 2
    title_weight: float = 0.7
    title_tokens: int = 30
    entity_dim: int = 100
    history_cap: int = 50
    snapshot_order_verified: bool = False
    recent_count: int = 10
    recent_hours: int = 24
    interest_count: int = 3
    ctr_strength: float = 20.0
    half_life_min_exposures: int = 100
    negative_count: int = 4
    update_mode: str = "prequential"
    context_features: bool = False
    parquet_batch_rows: int = 100000
    output_batch_rows: int = 100000

    def __post_init__(self):
        for key in (
            "text_dim",
            "max_features",
            "min_df",
            "title_tokens",
            "entity_dim",
            "history_cap",
            "recent_count",
            "recent_hours",
            "interest_count",
            "half_life_min_exposures",
            "negative_count",
            "parquet_batch_rows",
            "output_batch_rows",
        ):
            if getattr(self, key) <= 0:
                raise ValueError(f"{key} must be positive")
        if not 0 <= self.title_weight <= 1 or self.ctr_strength <= 0:
            raise ValueError("Invalid fusion/smoothing weights")
        if self.update_mode not in ("prequential", "frozen"):
            raise ValueError("update_mode must be prequential or frozen")

    def representation_fingerprint(self):
        fields = (
            "seed",
            "text_dim",
            "max_features",
            "min_df",
            "title_weight",
            "title_tokens",
            "entity_dim",
        )
        values = {name: getattr(self, name) for name in fields}
        return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

    def fingerprint(self):
        return hashlib.sha256(
            json.dumps(asdict(self), sort_keys=True).encode()
        ).hexdigest()

In [ ]:
DEFAULT_SETTINGS = Settings()

In [ ]:
# Edit settings here. Keep representation dimensions/vocabulary settings
# consistent across the notebooks that consume those artifacts.
SETTINGS = Settings(
    seed=4222,
    text_dim=64,
    max_features=50000,
    min_df=2,
    title_weight=0.7,
    title_tokens=30,
    entity_dim=100,
    history_cap=50,
    snapshot_order_verified=False,
    recent_count=10,
    recent_hours=24,
    interest_count=3,
    ctr_strength=20.0,
    half_life_min_exposures=100,
    negative_count=4,
    update_mode="prequential",
    context_features=False,
    parquet_batch_rows=100000,
    output_batch_rows=100000,
)
REBUILD = (
    False  # False: reuse verified outputs; True: regenerate this notebook's outputs
)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n")

In [ ]:
def normalise(values):
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    return np.divide(values, norms, out=np.zeros_like(values), where=norms > 1e-12)

In [ ]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))

In [ ]:
def manifest(root, cfg, sources, outputs, **extra):
    root = Path(root)
    try:
        revision = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        revision = None
    return dict(
        version=VERSION,
        created_utc=pd.Timestamp.now(tz="UTC").isoformat(),
        config=asdict(cfg),
        config_hash=cfg.fingerprint(),
        representation_config_hash=cfg.representation_fingerprint(),
        git_revision=revision,
        python=platform.python_version(),
        libraries={
            n: importlib.metadata.version(n)
            for n in ("numpy", "pandas", "pyarrow", "polars", "scikit-learn")
        },
        implementation_hashes={
            relative: sha256(root / relative)
            for relative in NOTEBOOK_PATHS
            if (root / relative).is_file()
        },
        sources={str(Path(p).relative_to(root)): sha256(p) for p in sources},
        outputs={str(Path(p).relative_to(root)): sha256(p) for p in outputs},
        output_bytes=sum((Path(p).stat().st_size for p in outputs)),
        **extra,
    )

In [ ]:
@contextmanager
def measure(root, stage, **details):
    import threading

    process = psutil.Process()
    peak = [process.memory_info().rss]
    stop = threading.Event()

    def monitor():
        while not stop.wait(0.05):
            peak[0] = max(peak[0], process.memory_info().rss)

    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        peak[0] = max(peak[0], process.memory_info().rss)
        stop.set()
        worker.join()
        path = Path(root) / "results" / "representation_cost.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        row = dict(
            stage=stage,
            wall_seconds=time.perf_counter() - start,
            sampled_peak_process_rss_mb=peak[0] / 1000000.0,
            platform=platform.platform(),
            cpu=platform.processor() or platform.machine(),
            logical_cpus=psutil.cpu_count(),
            ram_gb=psutil.virtual_memory().total / 1000000000.0,
            **details,
        )
        pd.DataFrame([row]).to_csv(
            path, mode="a", header=not path.exists(), index=False
        )

In [ ]:
def upsert_report(root, filename, frame, stage):
    path = Path(root) / "results" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame["stage"] = stage
    if path.exists():
        old = pd.read_csv(path)
        frame = pd.concat([old[old.stage != stage], frame], ignore_index=True)
    frame.to_csv(path, index=False)

In [ ]:
def require_inputs(root):
    p = Path(root) / "1_Data_Preparation" / "processed"
    files = [p / "news_catalogue.parquet", p / "users.parquet"]
    files += [
        p / f"{kind}_{split}.parquet"
        for kind in ("impressions", "candidates")
        for split in SPLITS
    ]
    missing = [str(f) for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(
            "Run 1_Data_Preparation/clean_and_validate_data.ipynb first. Missing:\n"
            + "\n".join(missing)
        )
    return p

### Article representation and availability logic

In [ ]:
def artifact_dir(root):
    path = Path(root) / "3_Representations" / "artifacts"
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
class Bundle:
    """Validated article-aligned arrays. Row 0 is padding/unknown, never a real article."""

    def __init__(self, root, cfg=DEFAULT_SETTINGS, verify_hashes=True):
        self.root = Path(root)
        self.cfg = cfg
        out = artifact_dir(root)
        index_hash = sha256(out / "article_index.parquet")
        for name in ("news", "entity", "availability"):
            meta = json.loads((out / f"{name}_manifest.json").read_text())
            if meta["representation_config_hash"] != cfg.representation_fingerprint():
                raise ValueError(
                    f"{name}: incompatible configuration; rebuild artifacts"
                )
            if meta.get("article_index_hash", index_hash) != index_hash:
                raise ValueError(
                    f"{name}: article map provenance mismatch; rebuild dependents"
                )
            if verify_hashes:
                for rel, expected in meta["outputs"].items():
                    if sha256(self.root / rel) != expected:
                        raise ValueError(f"Artifact hash mismatch: {rel}")
        self.index = pd.read_parquet(out / "article_index.parquet")
        self.lookup = dict(zip(self.index.news_code, self.index.article_row))
        self.n = len(self.index) + 1
        self.title = np.load(out / "title64.npy", mmap_mode="r")
        self.text = np.load(out / "text64.npy", mmap_mode="r")
        self.entity = np.load(out / "article_entity100.npy", mmap_mode="r")
        for array, dim in (
            (self.title, cfg.text_dim),
            (self.text, cfg.text_dim),
            (self.entity, cfg.entity_dim),
        ):
            norms = np.linalg.norm(array, axis=1)
            if (
                array.shape != (self.n, dim)
                or not np.isfinite(array).all()
                or np.any((norms > 1e-07) & (np.abs(norms - 1) > 0.0001))
                or (norms[0] != 0)
            ):
                raise ValueError(
                    "Invalid representation shape, coordinates, norm or padding"
                )
        self.meta = self.index.set_index("article_row").reindex(range(self.n))
        self.quality = pd.read_parquet(out / "entity_quality.parquet").set_index(
            "news_code"
        )
        self.availability = pd.read_parquet(
            out / "article_availability.parquet"
        ).set_index("news_code")
        if set(self.lookup) != set(self.availability.index):
            raise ValueError("Availability must cover exactly the prepared catalogue")
        self.category = np.r_[0, self.index.category_code.to_numpy(np.int32)]
        self.subcategory = np.r_[0, self.index.subcategory_code.to_numpy(np.int32)]

    def rows(self, codes):
        try:
            return np.asarray([self.lookup[int(c)] for c in codes], np.int32)
        except KeyError as exc:
            raise ValueError(f"Article absent from catalogue: {exc}") from exc

    def eligible(self, codes, t):
        return (
            self.availability.loc[list(codes), "available_from"] <= pd.Timestamp(t)
        ).to_numpy(bool)

### Historical state and user profiles

In [ ]:
HOUR = 3600000000000

In [ ]:
class PairStore:
    """Disk-backed user/article history; avoids a 97M-row Python dictionary.

    A fresh replay must use a fresh database. The persistent database is a scratch
    backend, not a complete replay checkpoint (profile/article state is separate).
    """

    def __init__(self, path=":memory:"):
        if str(path) != ":memory:":
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            if path.exists():
                raise FileExistsError(
                    f"Pair state already exists: {path}. Use a fresh replay path."
                )
        self.db = sqlite3.connect(str(path))
        self.db.execute("PRAGMA journal_mode=WAL")
        self.db.execute("PRAGMA synchronous=NORMAL")
        self.db.execute("PRAGMA cache_size=-32768")
        self.db.execute(
            "CREATE TABLE pairs(u INTEGER,a INTEGER,e INTEGER,c INTEGER,last INTEGER,PRIMARY KEY(u,a)) WITHOUT ROWID"
        )

    def lookup(self, users, articles):
        result = []
        pairs = list(zip(map(int, users), map(int, articles)))
        for start in range(0, len(pairs), 300):
            chunk = pairs[start : start + 300]
            sql = (
                "WITH wanted(i,u,a) AS (VALUES "
                + ",".join(["(?,?,?)"] * len(chunk))
                + ") SELECT COALESCE(p.e,0),COALESCE(p.c,0),p.last FROM wanted w LEFT JOIN pairs p ON p.u=w.u AND p.a=w.a ORDER BY w.i"
            )
            result.extend(
                self.db.execute(
                    sql, [v for i, pair in enumerate(chunk) for v in (i, *pair)]
                ).fetchall()
            )
        return result

    def update(self, users, articles, labels, t):
        grouped = Counter()
        for u, a, y in zip(users, articles, labels):
            grouped[int(u), int(a), "e"] += 1
            grouped[int(u), int(a), "c"] += int(y)
        keys = {(u, a) for u, a, _ in grouped}
        rows = [
            (u, a, grouped[u, a, "e"], grouped[u, a, "c"], int(t.value))
            for u, a in sorted(keys)
        ]
        self.db.executemany(
            "INSERT INTO pairs VALUES(?,?,?,?,?) ON CONFLICT(u,a) DO UPDATE SET e=e+excluded.e,c=c+excluded.c,last=excluded.last",
            rows,
        )
        self.db.commit()

    def close(self):
        self.db.close()

In [ ]:
@dataclass
class User:
    snapshot: np.ndarray
    snapshot_key: tuple
    observed: deque
    categories: Counter = field(default_factory=Counter)
    subcategories: Counter = field(default_factory=Counter)
    n_observed: int = 0
    last_impression: pd.Timestamp | None = None
    version: int = 0

In [ ]:
class TemporalState:
    def __init__(self, bundle, start, end, cfg=DEFAULT_SETTINGS, pair_path=":memory:"):
        self.bundle = bundle
        self.cfg = cfg
        self.start = pd.Timestamp(start)
        self.end = pd.Timestamp(end)
        self.start_hour = self.start.value // HOUR
        hours = int(self.end.value // HOUR - self.start_hour) + 1
        if hours < 1:
            raise ValueError("End before start")
        self.exposures = np.zeros(bundle.n, np.int64)
        self.clicks = np.zeros(bundle.n, np.int64)
        self.first = np.full(bundle.n, -1, np.int64)
        self.hour_exposures = np.zeros((bundle.n, hours), np.int32)
        self.hour_clicks = np.zeros((bundle.n, hours), np.int32)
        self.cat = np.concatenate(
            [
                ["<PAD>"],
                bundle.index.category.fillna("<MISSING>").astype(str).to_numpy(),
            ]
        )
        self.subcat = np.concatenate(
            [
                ["<PAD>"],
                bundle.index.subcategory.fillna("<MISSING>").astype(str).to_numpy(),
            ]
        )
        self.users = {}
        self.profiles = {}
        self.pairs = PairStore(pair_path)
        self.total_exposures = 0
        self.total_clicks = 0
        self.last_update = None
        self.active_time = None

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple((int(token.removeprefix("N")) for token in text.split()))
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=self.cfg.history_cap))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.sort(user.snapshot)
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        text, valid_text = normalised_mean(self.bundle.text, history)
        entity, valid_entity = normalised_mean(self.bundle.entity, history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            rows = history[self.cat[history] == cat]
            vector, valid = normalised_mean(self.bundle.title, rows)
            if valid and np.linalg.norm(vector) > 0:
                interests.append(vector)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(history)),
            "valid_entity_fraction": valid_entity / max(1, len(history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def half_life(self, rows, t):
        t = pd.Timestamp(t)
        recent_end = int(t.value // HOUR - self.start_hour)
        recent_start = recent_end - 6
        result = []
        for row in rows:
            first = self.first[row]
            if first < 0:
                result.append((0.0, False, "no_prior_exposure", 0, 0))
                continue
            early_start = int((first + HOUR - 1) // HOUR - self.start_hour)
            early_end = early_start + 6
            if recent_start < early_end:
                result.append((0.0, False, "young_or_overlapping", 0, 0))
                continue
            early_e = int(self.hour_exposures[row, early_start:early_end].sum())
            recent_e = int(self.hour_exposures[row, recent_start:recent_end].sum())
            if min(early_e, recent_e) < self.cfg.half_life_min_exposures:
                result.append((0.0, False, "low_support", early_e, recent_e))
                continue
            early_c = int(self.hour_clicks[row, early_start:early_end].sum())
            recent_c = int(self.hour_clicks[row, recent_start:recent_end].sum())
            early = (early_c + 1) / (early_e + 2)
            recent = (recent_c + 1) / (recent_e + 2)
            if recent >= early:
                result.append((0.0, False, "no_decline", early_e, recent_e))
                continue
            estimate = np.log(2) * (recent_start - early_start) / np.log(early / recent)
            ok = bool(np.isfinite(estimate) and estimate > 0)
            result.append(
                (
                    float(estimate) if ok else 0.0,
                    ok,
                    "available" if ok else "invalid",
                    early_e,
                    recent_e,
                )
            )
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

    def close(self):
        self.pairs.close()

### Ranking feature definitions

In [ ]:
GROUPS = {
    "content": [
        "category_code",
        "subcategory_code",
        "title_word_count",
        "abstract_word_count",
        "has_abstract",
        "n_annotated_entities",
        "entity_coverage",
        "article_has_text",
        "article_has_entity",
    ],
    "history": [
        "log_history_length",
        "retained_history_length",
        "valid_text_history_fraction",
        "valid_entity_history_fraction",
        "no_supplied_history",
        "no_causal_history",
        "no_text_profile",
        "no_entity_profile",
        "no_recent_profile",
    ],
    "match": [
        "category_affinity",
        "subcategory_affinity",
        "seen_category",
        "text_cosine",
        "entity_cosine",
        "long_term_cosine",
        "recent_cosine",
        "text_match_valid",
        "entity_match_valid",
        "long_term_match_valid",
        "recent_match_valid",
    ],
    "repeat": [
        "log_prior_exposure_count",
        "log_prior_nonclick_count",
        "log_prior_pair_click_count",
        "hours_since_last_exposure",
        "no_prior_pair_exposure",
        "previously_clicked",
    ],
    "popularity": [
        "log_prior_article_exposures",
        "log_prior_article_clicks",
        "smoothed_prior_article_ctr",
        "article_low_support",
        "population_startup",
    ],
    "freshness": [
        "article_age_hours",
        "no_prior_article_exposure",
        "age_left_censored",
    ],
    "multi_interest": [
        "multi_interest_match_score",
        "interest_group_count",
        "selected_interest_group_count",
        "multi_interest_valid",
    ],
    "half_life": [
        "estimated_article_half_life_hours",
        "has_half_life",
        "stale_after_grace",
    ],
    "request_context": [
        "day_part",
        "weekday",
        "hours_since_previous_impression",
        "first_visit",
    ],
    "slate_context": ["personalized_relative_advantage", "slate_context_valid"],
}

In [ ]:
FORMULAS = {
    "category_affinity": "full prior clicked category count / full history count; zero if empty",
    "subcategory_affinity": "full prior clicked subcategory count / full history count; zero if empty",
    "seen_category": "prior candidate-category count > 0",
    "smoothed_prior_article_ctr": "(prior article clicks + strength * expanding Beta(1,1) global CTR) / (prior article exposures + strength)",
    "article_age_hours": "hours since first prior observed exposure; zero with flag if none",
    "estimated_article_half_life_hours": "ln(2) * completed 6h window-centre gap / ln(early CTR / recent CTR); zero plus mask when unsupported",
    "multi_interest_match_score": "max title64 cosine over up to three valid category interest vectors",
    "personalized_relative_advantage": "candidate multi-interest match minus actual candidate-set median; zero if context invalid",
}

In [ ]:
def feature_schema(cfg=DEFAULT_SETTINGS):
    features = []
    for group, names in GROUPS.items():
        if group == "slate_context" and (not cfg.context_features):
            continue
        for name in names:
            features.append(
                {
                    "name": name,
                    "dtype": "float32",
                    "group": group,
                    "missing_value": 0,
                    "definition": FORMULAS.get(name, name.replace("_", " ")),
                    "cutoff": "static train-fitted content or events strictly before request timestamp",
                    "preprocessing": "log1p already applied"
                    if name.startswith("log_")
                    else "none; fit any scaler on training rows only",
                    "consumer": "ranker; slate context requires complete actual candidate set"
                    if group == "slate_context"
                    else "ranker",
                }
            )
    return {
        "version": "ranking-features-v1",
        "features": features,
        "model_columns": [f["name"] for f in features],
        "keys": ["impression_uid", "news_code"],
        "diagnostics": [
            "user_code",
            "article_row",
            "timestamp",
            "state_version",
            "candidate_source",
            "eligible",
            "history_length",
            "prior_article_exposures",
            "prior_article_clicks",
            "prior_exposure_count",
            "half_life_reason",
            "half_life_early_support",
            "half_life_recent_support",
        ],
        "excluded": [
            "clicked",
            "position",
            "slate_len",
            "n_clicks",
            "n_negatives",
            "split",
            "user_code",
            "news_code",
            "impression_uid",
        ],
        "update_mode": cfg.update_mode,
        "config_hash": cfg.fingerprint(),
    }

In [ ]:
def build_features(
    requests,
    candidate_pairs,
    state_before_t,
    representation_bundle,
    feature_settings=DEFAULT_SETTINGS,
    candidate_source="retrieved",
):
    """Same input contract for logged or unlabelled retrieved candidates.

    Caller begins one complete timestamp, builds all feature requests, then calls
    finish with logged feedback. This function neither reads labels nor mutates state.
    """
    state = state_before_t
    bundle = representation_bundle
    cfg = feature_settings
    if state.cfg.fingerprint() != cfg.fingerprint():
        raise ValueError("State and feature configuration differ")
    if candidate_source not in ("logged", "retrieved"):
        raise ValueError("candidate_source must be logged or retrieved")
    if (
        requests.impression_uid.duplicated().any()
        or candidate_pairs.duplicated(["impression_uid", "news_code"]).any()
    ):
        raise ValueError("Duplicate request or candidate key")
    if not len(requests) or requests.timestamp.nunique() != 1:
        raise ValueError("Feature call requires one nonempty timestamp batch")
    t = pd.Timestamp(requests.timestamp.iloc[0])
    if state.active_time != t:
        raise ValueError("Call state.begin for this timestamp first")
    if not set(candidate_pairs.impression_uid) <= set(requests.impression_uid):
        raise ValueError("Candidate has no request")
    if not len(candidate_pairs):
        raise ValueError("Empty candidate batch")
    pairs = candidate_pairs[["impression_uid", "news_code"]].merge(
        requests[["impression_uid", "user_code"]],
        on="impression_uid",
        validate="many_to_one",
        sort=False,
    )
    rows = bundle.rows(pairs.news_code)
    eligible = bundle.eligible(pairs.news_code, t)
    if not eligible.all():
        raise ValueError("Candidate catalogue contains future/unavailable articles")
    prior_pairs = state.pairs.lookup(pairs.user_code, rows)
    unique_rows, inverse = np.unique(rows, return_inverse=True)
    unique_half = state.half_life(unique_rows, t)
    half = [unique_half[i] for i in inverse]
    global_prior = (state.total_clicks + 1) / (state.total_exposures + 2)
    n = len(pairs)
    metadata = bundle.meta.loc[rows].reset_index(drop=True)
    quality = bundle.quality.loc[pairs.news_code.to_numpy()].reset_index(drop=True)
    article_e = state.exposures[rows]
    article_c = state.clicks[rows]
    first = state.first[rows]
    seen = first >= 0
    pair_e = np.asarray([x[0] for x in prior_pairs], np.int64)
    pair_c = np.asarray([x[1] for x in prior_pairs], np.int64)
    pair_last = np.asarray(
        [-1 if x[2] is None else x[2] for x in prior_pairs], np.int64
    )
    h = np.asarray([x[0] for x in half], np.float64)
    has_h = np.asarray([x[1] for x in half], bool)
    text_valid = np.linalg.norm(bundle.text[rows], axis=1) > 0
    entity_valid = np.linalg.norm(bundle.entity[rows], axis=1) > 0
    title_valid = np.linalg.norm(bundle.title[rows], axis=1) > 0
    ages = np.where(seen, (t.value - first) / HOUR, 0)
    values = {
        "impression_uid": pairs.impression_uid.to_numpy(np.int64),
        "news_code": pairs.news_code.to_numpy(np.int64),
        "user_code": pairs.user_code.to_numpy(np.int64),
        "article_row": rows,
        "timestamp": t,
        "candidate_source": candidate_source,
        "eligible": True,
        "category_code": metadata.category_code.to_numpy(),
        "subcategory_code": metadata.subcategory_code.to_numpy(),
        "title_word_count": metadata.title_word_count.to_numpy(),
        "abstract_word_count": metadata.abstract_word_count.to_numpy(),
        "has_abstract": metadata.has_abstract.to_numpy(),
        "n_annotated_entities": quality.n_annotated_entities.to_numpy(),
        "entity_coverage": quality.entity_coverage.to_numpy(),
        "article_has_text": text_valid,
        "article_has_entity": entity_valid,
        "prior_exposure_count": pair_e,
        "log_prior_exposure_count": np.log1p(pair_e),
        "log_prior_nonclick_count": np.log1p(pair_e - pair_c),
        "log_prior_pair_click_count": np.log1p(pair_c),
        "hours_since_last_exposure": np.where(
            pair_last >= 0, (t.value - pair_last) / HOUR, 0
        ),
        "no_prior_pair_exposure": pair_last < 0,
        "prior_article_exposures": article_e,
        "prior_article_clicks": article_c,
        "log_prior_article_exposures": np.log1p(article_e),
        "log_prior_article_clicks": np.log1p(article_c),
        "smoothed_prior_article_ctr": (article_c + cfg.ctr_strength * global_prior)
        / (article_e + cfg.ctr_strength),
        "article_low_support": article_e < 100,
        "population_startup": state.total_exposures == 0,
        "article_age_hours": ages,
        "no_prior_article_exposure": ~seen,
        "age_left_censored": first == state.start.value,
        "estimated_article_half_life_hours": h,
        "has_half_life": has_h,
        "stale_after_grace": has_h & (ages > h + 6),
        "half_life_reason": [x[2] for x in half],
        "half_life_early_support": [x[3] for x in half],
        "half_life_recent_support": [x[4] for x in half],
        "day_part": t.hour // 6,
        "weekday": t.weekday(),
    }
    user_columns = [
        "state_version",
        "history_length",
        "log_history_length",
        "retained_history_length",
        "valid_text_history_fraction",
        "valid_entity_history_fraction",
        "no_supplied_history",
        "no_causal_history",
        "no_text_profile",
        "no_entity_profile",
        "no_recent_profile",
        "category_affinity",
        "subcategory_affinity",
        "seen_category",
        "text_cosine",
        "entity_cosine",
        "long_term_cosine",
        "recent_cosine",
        "previously_clicked",
        "multi_interest_match_score",
        "interest_group_count",
        "selected_interest_group_count",
        "multi_interest_valid",
        "hours_since_previous_impression",
        "first_visit",
        "text_match_valid",
        "entity_match_valid",
        "long_term_match_valid",
        "recent_match_valid",
    ]
    values.update({name: np.zeros(n, np.float64) for name in user_columns})
    for user_code, positions in pairs.groupby("user_code", sort=False).indices.items():
        idx = np.asarray(positions)
        article_rows = rows[idx]
        user = state.users[int(user_code)]
        p = state.profile(user_code, t)
        total = p["history_length"]
        category_counts = np.asarray(
            [user.categories[c] for c in state.cat[article_rows]]
        )
        subcategory_counts = np.asarray(
            [user.subcategories[c] for c in state.subcat[article_rows]]
        )
        dynamic = {
            "state_version": p["state_version"],
            "history_length": total,
            "log_history_length": np.log1p(total),
            "retained_history_length": p["retained_length"],
            "valid_text_history_fraction": p["valid_text_fraction"],
            "valid_entity_history_fraction": p["valid_entity_fraction"],
            "no_supplied_history": p["no_supplied_history"],
            "no_causal_history": total == 0,
            "no_text_profile": not p["has_text"],
            "no_entity_profile": not p["has_entity"],
            "no_recent_profile": not p["has_recent"],
            "category_affinity": category_counts / max(1, total),
            "subcategory_affinity": subcategory_counts / max(1, total),
            "seen_category": category_counts > 0,
            "text_cosine": bundle.text[article_rows] @ p["text"],
            "entity_cosine": bundle.entity[article_rows] @ p["entity"],
            "long_term_cosine": bundle.text[article_rows] @ p["long_text"],
            "recent_cosine": bundle.text[article_rows] @ p["recent_text"],
            "text_match_valid": p["has_text"] & text_valid[idx],
            "entity_match_valid": p["has_entity"] & entity_valid[idx],
            "long_term_match_valid": bool(np.linalg.norm(p["long_text"]) > 0)
            & text_valid[idx],
            "recent_match_valid": p["has_recent"] & text_valid[idx],
            "previously_clicked": (pair_c[idx] > 0)
            | np.isin(article_rows, user.snapshot),
            "multi_interest_match_score": np.max(
                bundle.title[article_rows] @ p["interests"].T, axis=1
            )
            if p["group_count"]
            else np.zeros(len(idx)),
            "interest_group_count": p["group_count"],
            "selected_interest_group_count": p["selected_group_count"],
            "multi_interest_valid": bool(p["group_count"]) & title_valid[idx],
            "hours_since_previous_impression": (
                t - user.last_impression
            ).total_seconds()
            / 3600
            if user.last_impression is not None
            else 0,
            "first_visit": user.last_impression is None,
        }
        for name, value in dynamic.items():
            values[name][idx] = value
    result = pd.DataFrame(values)
    result["state_version"] = result.state_version.astype(np.int64)
    result["history_length"] = result.history_length.astype(np.int64)
    if cfg.context_features:
        grouped = result.groupby("impression_uid")
        valid = grouped.multi_interest_valid.transform("all")
        result["slate_context_valid"] = valid
        result["personalized_relative_advantage"] = (
            result.multi_interest_match_score
            - grouped.multi_interest_match_score.transform("median")
        ).where(valid, 0)
    columns = feature_schema(cfg)["model_columns"]
    result[columns] = result[columns].astype(np.float32)
    if not np.isfinite(result[columns].to_numpy()).all():
        raise ValueError("Nonfinite model features")
    return result

In [ ]:
def sample_training_groups(candidates, cfg=DEFAULT_SETTINGS):
    """Reproducible per-positive exposed negatives, independent of input row order."""
    groups = []
    selected = set()
    for uid, slate in candidates.groupby("impression_uid", sort=True):
        if slate.news_code.duplicated().any() or not slate.clicked.isin([0, 1]).all():
            raise ValueError("Sampling requires unique binary-labelled logged pairs")
        positives = sorted(map(int, slate.loc[slate.clicked.eq(1), "news_code"]))
        negatives = np.sort(
            slate.loc[slate.clicked.eq(0), "news_code"].to_numpy(np.int64)
        )
        if not positives:
            raise ValueError("Logged training impression has no positives")
        for positive in positives:
            seed = int.from_bytes(
                hashlib.sha256(f"{cfg.seed}:{int(uid)}:{positive}".encode()).digest()[
                    :8
                ],
                "little",
            )
            rng = np.random.default_rng(seed)
            sampled = sorted(
                map(
                    int,
                    rng.choice(
                        negatives,
                        min(cfg.negative_count, len(negatives)),
                        replace=False,
                    ),
                )
            )
            groups.append(
                {
                    "impression_uid": int(uid),
                    "positive_news_code": positive,
                    "negative_news_codes": sampled,
                }
            )
            selected.add((int(uid), positive))
            selected.update(((int(uid), n) for n in sampled))
    return (
        pd.DataFrame(
            groups,
            columns=["impression_uid", "positive_news_code", "negative_news_codes"],
        ),
        selected,
    )

### Reading and writing the datasets

In [ ]:
def timestamp_groups(path, columns, batch_size=100000):
    """Bounded parquet reads retaining complete timestamp groups across row groups."""
    pending = None
    last = None
    for batch in pq.ParquetFile(path).iter_batches(
        batch_size=batch_size, columns=columns
    ):
        frame = batch.to_pandas()
        frame["timestamp"] = pd.to_datetime(frame.timestamp).astype("datetime64[ns]")
        if frame.timestamp.isna().any() or not frame.timestamp.is_monotonic_increasing:
            raise ValueError(f"Unsorted or missing timestamps: {path}")
        if last is not None and frame.timestamp.iloc[0] < last:
            raise ValueError(f"Parquet row groups out of order: {path}")
        last = frame.timestamp.iloc[-1]
        if pending is not None:
            frame = pd.concat([pending, frame], ignore_index=True)
        tail = frame.timestamp.iloc[-1]
        ready = frame[frame.timestamp != tail]
        pending = frame[frame.timestamp == tail]
        for t, group in ready.groupby("timestamp", sort=False):
            yield (pd.Timestamp(t), group.reset_index(drop=True))
    if pending is not None:
        yield (pd.Timestamp(pending.timestamp.iloc[0]), pending.reset_index(drop=True))

In [ ]:
def replay_batches(root, cfg=DEFAULT_SETTINGS, splits=SPLITS):
    processed = require_inputs(root)
    streams = []
    for split in splits:
        for kind, columns in (
            ("impressions", ["impression_uid", "user_code", "timestamp", "history"]),
            (
                "candidates",
                ["impression_uid", "user_code", "timestamp", "news_code", "clicked"],
            ),
        ):
            streams.append(
                (
                    kind,
                    split,
                    iter(
                        timestamp_groups(
                            processed / f"{kind}_{split}.parquet",
                            columns,
                            cfg.parquet_batch_rows,
                        )
                    ),
                )
            )
    heap = []
    for i, (_, _, stream) in enumerate(streams):
        first = next(stream, None)
        if first is not None:
            heapq.heappush(heap, (first[0], i, first[1]))
    while heap:
        t = heap[0][0]
        requests = []
        events = []
        while heap and heap[0][0] == t:
            _, i, frame = heapq.heappop(heap)
            kind, split, stream = streams[i]
            frame = frame.copy()
            frame["split"] = split
            (requests if kind == "impressions" else events).append(frame)
            nxt = next(stream, None)
            if nxt is not None:
                heapq.heappush(heap, (nxt[0], i, nxt[1]))
        if not requests or not events:
            raise ValueError(f"Unpaired requests/events at {t}")
        req = pd.concat(requests, ignore_index=True)
        ev = pd.concat(events, ignore_index=True)
        if (
            req.impression_uid.duplicated().any()
            or ev.duplicated(["impression_uid", "news_code"]).any()
        ):
            raise ValueError("Duplicate request/pair in timestamp stream")
        if set(ev.impression_uid) != set(req.impression_uid):
            raise ValueError("Every request must have its logged candidate slate")
        owners = req.set_index("impression_uid").user_code
        if not np.array_equal(
            ev.user_code.to_numpy(), ev.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Candidate/request user mismatch")
        yield (t, req, ev)

In [ ]:
class PartitionWriter:
    """Write typed Parquet parts; keep only a bounded number of pending rows."""

    def __init__(self, path, rows=100000, schema=None):
        self.path = Path(path)
        self.path.mkdir(parents=True, exist_ok=True)
        self.frames = []
        self.pending = 0
        self.count = 0
        self.part = 0
        self.threshold = rows
        self.schema = schema

    def add(self, frame):
        if not len(frame):
            return
        self.frames.append(frame)
        self.pending += len(frame)
        self.count += len(frame)
        if self.pending >= self.threshold:
            self.flush()

    def flush(self):
        if not self.pending:
            return
        table = pa.Table.from_pandas(
            pd.concat(self.frames, ignore_index=True), preserve_index=False
        )
        if self.schema is None:
            self.schema = table.schema
        table = table.cast(self.schema)
        pq.write_table(
            table, self.path / f"part-{self.part:05d}.parquet", compression="zstd"
        )
        self.part += 1
        self.frames = []
        self.pending = 0

    def close(self):
        self.flush()

In [ ]:
def time_horizon(root, splits=SPLITS):
    import polars as pl

    p = require_inputs(root)
    ranges = [
        pl.scan_parquet(p / f"impressions_{s}.parquet")
        .select(
            pl.col("timestamp").min().alias("min"),
            pl.col("timestamp").max().alias("max"),
        )
        .collect()
        .row(0)
        for s in splits
    ]
    return (min((x[0] for x in ranges)), max((x[1] for x in ranges)))

In [ ]:
def _profile_references(requests, state, t):
    refs = []
    for row in requests.itertuples(index=False):
        p = state.profile(row.user_code, t)
        refs.append(
            {
                "impression_uid": int(row.impression_uid),
                "user_code": int(row.user_code),
                "timestamp": t,
                "state_version": p["state_version"],
                "history_rows": p["history_rows"].tolist(),
                "history_mask": p["history_mask"].tolist(),
                "history_length": p["history_length"],
                "retained_length": p["retained_length"],
                "interest_group_count": p["group_count"],
                "selected_interest_categories": p["selected_interest_categories"],
                "recent_rows": p["recent_rows"],
                "no_supplied_history": p["no_supplied_history"],
                "has_text": p["has_text"],
                "has_entity": p["has_entity"],
            }
        )
    return pd.DataFrame(refs)

In [ ]:
def build_ranking_dataset(
    root,
    cfg=DEFAULT_SETTINGS,
    *,
    target_split=None,
    max_timestamps=None,
    overwrite=False,
):
    """Replay complete historical logs; sample only the written training examples.

    max_timestamps is a chronological smoke run starting at training start, never
    an arbitrary slice without state. Smoke output has a separate run directory.
    Frozen mode requires one target split; earlier splits still update state.
    """
    root = Path(root)
    if max_timestamps is not None and max_timestamps < 1:
        raise ValueError("max_timestamps must be positive or None")
    processed = require_inputs(root)
    if cfg.update_mode == "frozen" and target_split not in SPLITS:
        raise ValueError("Frozen mode requires target_split")
    if target_split is not None and target_split not in SPLITS:
        raise ValueError("Unknown target split")
    bundle = Bundle(root, cfg)
    import polars as pl

    train_users = set(
        pl.scan_parquet(processed / "impressions_train.parquet")
        .select("user_code")
        .unique()
        .collect()["user_code"]
        .to_list()
    )
    train_articles = set(
        pl.scan_parquet(processed / "candidates_train.parquet")
        .select("news_code")
        .unique()
        .collect()["news_code"]
        .to_list()
    )
    splits = (
        SPLITS if target_split is None else SPLITS[: SPLITS.index(target_split) + 1]
    )
    name = f"{cfg.update_mode}-{target_split or 'all'}" + (
        f"-smoke-{max_timestamps}" if max_timestamps else ""
    )
    out = root / "4_Feature_Engineering" / "artifacts" / name
    if out.exists():
        if not overwrite:
            raise FileExistsError(
                f"Output exists: {out}; use overwrite=True to rebuild this run"
            )
        shutil.rmtree(out)
    out.mkdir(parents=True)
    write_json(out / "feature_schema.json", feature_schema(cfg))
    write_json(
        out / "run_status.json",
        {"status": "running", "full_run": max_timestamps is None},
    )
    write_splits = [target_split] if target_split else list(splits)
    coverage = Counter()
    quality = {}
    generated = Counter()
    selected_counts = Counter()
    feature_writers = {
        s: PartitionWriter(out / f"ranking_features_{s}", cfg.output_batch_rows)
        for s in write_splits
    }
    target_writers = {
        s: PartitionWriter(out / f"labels_{s}", cfg.output_batch_rows)
        for s in write_splits
    }
    reference_schema = pa.schema(
        [
            ("impression_uid", pa.int64()),
            ("user_code", pa.int64()),
            ("timestamp", pa.timestamp("ns")),
            ("state_version", pa.int64()),
            ("history_rows", pa.list_(pa.int32())),
            ("history_mask", pa.list_(pa.bool_())),
            ("history_length", pa.int64()),
            ("retained_length", pa.int64()),
            ("interest_group_count", pa.int64()),
            ("selected_interest_categories", pa.list_(pa.string())),
            ("recent_rows", pa.list_(pa.int32())),
            ("no_supplied_history", pa.bool_()),
            ("has_text", pa.bool_()),
            ("has_entity", pa.bool_()),
        ]
    )
    ref_writers = {
        s: PartitionWriter(
            out / f"request_profiles_{s}", cfg.output_batch_rows, reference_schema
        )
        for s in write_splits
    }
    group_writer = PartitionWriter(
        out / "training_negative_groups",
        cfg.output_batch_rows,
        pa.schema(
            [
                ("impression_uid", pa.int64()),
                ("positive_news_code", pa.int64()),
                ("negative_news_codes", pa.list_(pa.int64())),
            ]
        ),
    )
    state = TemporalState(
        bundle, *time_horizon(root, splits), cfg, pair_path=out / "pair_history.sqlite"
    )
    written = 0
    cutoff = None
    model_columns = feature_schema(cfg)["model_columns"]
    try:
        with measure(root, "ranking_features"):
            for t, requests, events in replay_batches(root, cfg, splits):
                state.begin(requests, t)
                for split in write_splits:
                    req = requests[requests.split.eq(split)]
                    ev = events[events.split.eq(split)]
                    if not len(req):
                        continue
                    f = build_features(req, ev, state, bundle, cfg, "logged")
                    generated[split] += len(f)
                    clicked = (
                        ev.set_index(["impression_uid", "news_code"])
                        .clicked.reindex(
                            pd.MultiIndex.from_frame(f[["impression_uid", "news_code"]])
                        )
                        .to_numpy()
                        == 1
                    )
                    new_user = np.fromiter(
                        (int(u) not in train_users for u in f.user_code), bool, len(f)
                    )
                    new_article = np.fromiter(
                        (int(a) not in train_articles for a in f.news_code),
                        bool,
                        len(f),
                    )
                    cohorts = (
                        ("shown_exposures", np.ones(len(f), bool)),
                        ("clicked_exposures", clicked),
                        ("new_users_vs_train", new_user),
                        ("returning_users_vs_train", ~new_user),
                        ("articles_unexposed_in_train", new_article),
                        ("no_causal_history", f.no_causal_history.to_numpy() > 0),
                    )
                    for rep, col in (
                        ("text", "article_has_text"),
                        ("entity", "article_has_entity"),
                        ("text_profile", "no_text_profile"),
                        ("entity_profile", "no_entity_profile"),
                    ):
                        valid = (
                            f[col].to_numpy() > 0
                            if rep in ("text", "entity")
                            else f[col].to_numpy() == 0
                        )
                        for cohort, mask in cohorts:
                            coverage[split, rep, cohort, "n"] += int(mask.sum())
                            coverage[split, rep, cohort, "k"] += int(valid[mask].sum())
                    matrix = f[model_columns].to_numpy()
                    zero = (matrix == 0).sum(axis=0)
                    minimum = matrix.min(axis=0)
                    maximum = matrix.max(axis=0)
                    for j, col in enumerate(model_columns):
                        old = quality.setdefault(
                            (split, col),
                            {
                                "n": 0,
                                "zero": 0,
                                "min": float("inf"),
                                "max": -float("inf"),
                            },
                        )
                        old["n"] += len(f)
                        old["zero"] += int(zero[j])
                        old["min"] = min(old["min"], float(minimum[j]))
                        old["max"] = max(old["max"], float(maximum[j]))
                    ref_writers[split].add(_profile_references(req, state, t))
                    if split == "train":
                        groups, selected = sample_training_groups(ev, cfg)
                        group_writer.add(groups)
                        keep = np.array(
                            [
                                (int(u), int(a)) in selected
                                for u, a in zip(f.impression_uid, f.news_code)
                            ]
                        )
                        selected_counts["all_negatives"] += int(ev.clicked.eq(0).sum())
                        kept_labels = ev.set_index(
                            ["impression_uid", "news_code"]
                        ).clicked.reindex(
                            pd.MultiIndex.from_frame(
                                f.loc[keep, ["impression_uid", "news_code"]]
                            )
                        )
                        selected_counts["kept_unique_negatives"] += int(
                            kept_labels.eq(0).sum()
                        )
                        selected_counts["positive_groups"] += len(groups)
                        selected_counts["negative_group_memberships"] += sum(
                            map(len, groups.negative_news_codes)
                        )
                        f = f.loc[keep].reset_index(drop=True)
                    labels = f[["impression_uid", "news_code"]].merge(
                        ev[["impression_uid", "news_code", "clicked"]],
                        validate="one_to_one",
                    )
                    feature_writers[split].add(f)
                    target_writers[split].add(labels)
                feedback = set(splits)
                if cfg.update_mode == "frozen":
                    feedback.discard(target_split)
                state.finish(requests, events, t, feedback)
                written += 1
                cutoff = t
                if written % 10000 == 0:
                    print(
                        f"{written:,} timestamp batches; cutoff {t}; emitted {sum((w.count for w in feature_writers.values())):,} pairs",
                        flush=True,
                    )
                if max_timestamps is not None and written >= max_timestamps:
                    break
            for writer in [
                *feature_writers.values(),
                *target_writers.values(),
                *ref_writers.values(),
                group_writer,
            ]:
                writer.close()
        coverage_rows = []
        for split, rep, cohort, kind in sorted(coverage):
            if kind != "n":
                continue
            n = coverage[split, rep, cohort, "n"]
            k = coverage[split, rep, cohort, "k"]
            coverage_rows.append(
                {
                    "split": split,
                    "representation": rep,
                    "cohort": cohort,
                    "denominator": n,
                    "valid": k,
                    "fallback": n - k,
                    "valid_fraction": k / max(1, n),
                }
            )
        pd.DataFrame(coverage_rows).to_csv(out / "coverage.csv", index=False)
        qrows = [
            dict(
                split=s,
                feature=f,
                **values,
                zero_fraction=values["zero"] / values["n"],
                nonfinite_count=0,
            )
            for (s, f), values in quality.items()
        ]
        pd.DataFrame(qrows).to_csv(out / "feature_quality.csv", index=False)
        write_json(
            out / "sampling_summary.json",
            dict(
                selected_counts,
                discarded_unique_negative_fraction=1
                - selected_counts["kept_unique_negatives"]
                / max(1, selected_counts["all_negatives"]),
            ),
        )
        sources = [
            processed / f"{kind}_{s}.parquet"
            for kind in ("impressions", "candidates")
            for s in splits
        ]
        sources += [
            artifact_dir(root) / f"{x}_manifest.json"
            for x in ("news", "entity", "availability")
        ]
        state.close()
        state = None
        for scratch in out.glob("pair_history.sqlite*"):
            scratch.unlink()
        costs = pd.read_csv(root / "results" / "representation_cost.csv")
        latest = costs[costs.stage.eq("ranking_features")].iloc[-1].to_dict()
        cost = pd.DataFrame(
            [
                dict(
                    latest,
                    run=name,
                    artifact_bytes=sum(
                        (p.stat().st_size for p in out.rglob("*.parquet"))
                    ),
                    emitted_rows=sum((w.count for w in feature_writers.values())),
                    timestamp_batches=written,
                )
            ]
        )
        cost.to_csv(out / "feature_cost.csv", index=False)
        outputs = list(out.rglob("*.parquet")) + [
            out / "feature_schema.json",
            out / "coverage.csv",
            out / "feature_quality.csv",
            out / "sampling_summary.json",
            out / "feature_cost.csv",
        ]
        write_json(
            out / "run_manifest.json",
            manifest(
                root,
                cfg,
                sources,
                outputs,
                full_run=max_timestamps is None,
                last_scored_timestamp=cutoff,
                timestamp_batches=written,
                rows_before_sampling=dict(generated),
                rows_written={s: w.count for s, w in feature_writers.items()},
                pair_state_backend="sqlite scratch; not resumable",
                half_life_version="completed disjoint 6h windows; Beta(1,1) smoothing",
                history_order="verified sequence"
                if cfg.snapshot_order_verified
                else "deterministic snapshot subset; no recency claim",
            ),
        )
        write_json(
            out / "run_status.json",
            {
                "status": "complete",
                "full_run": max_timestamps is None,
                "last_scored_timestamp": cutoff,
            },
        )
        if max_timestamps is None:
            upsert_report(
                root, "representation_coverage.csv", pd.DataFrame(coverage_rows), name
            )
            upsert_report(root, "feature_quality.csv", pd.DataFrame(qrows), name)
            upsert_report(root, "feature_cost.csv", cost, name)
    except Exception as exc:
        write_json(
            out / "run_status.json",
            {"status": "failed", "error": str(exc), "last_finished_timestamp": cutoff},
        )
        raise
    finally:
        if state is not None:
            state.close()
    return out

### Loading saved outputs

In [ ]:
def open_feature_run(root, run_dir, *, allow_smoke=False, verify_hashes=True):
    """Return schema/manifest and Arrow datasets without loading all features in RAM."""
    root = Path(root)
    path = Path(run_dir)
    status = json.loads((path / "run_status.json").read_text())
    meta = json.loads((path / "run_manifest.json").read_text())
    if status["status"] != "complete":
        raise ValueError("Feature run is incomplete")
    if not meta["full_run"] and (not allow_smoke):
        raise ValueError("Smoke output is not a full training/evaluation dataset")
    if verify_hashes:
        for rel, expected in meta["outputs"].items():
            if sha256(root / rel) != expected:
                raise ValueError(f"Feature artifact mismatch: {rel}")
        for rel, expected in meta["sources"].items():
            if not (root / rel).exists() or sha256(root / rel) != expected:
                raise ValueError(f"Input or representation dependency changed: {rel}")
    schema = json.loads((path / "feature_schema.json").read_text())
    datasets = {}
    for split in meta["rows_written"]:
        p = path / f"ranking_features_{split}"
        if list(p.glob("*.parquet")):
            datasets[split] = ds.dataset(p, format="parquet")
    return {"schema": schema, "manifest": meta, "datasets": datasets}

In [ ]:
def reconstruct_profile(reference, bundle):
    """Recover capped causal vectors/interest groups from a stored request reference.

    Long-term supplied profiles use the separate supplied-profile matrix. No labels
    or future user state are needed for this reconstruction.
    """
    history = np.asarray(reference["history_rows"], np.int32)
    mask = np.asarray(reference["history_mask"], bool)
    history = history[mask]
    text, _ = normalised_mean(bundle.text, history)
    entity, _ = normalised_mean(bundle.entity, history)
    recent, _ = normalised_mean(bundle.text, reference["recent_rows"])
    categories = bundle.meta.category.fillna("<MISSING>").astype(str).to_numpy()
    interests = []
    for category in reference["selected_interest_categories"]:
        rows = history[categories[history] == category]
        vector, count = normalised_mean(bundle.title, rows)
        if count and np.linalg.norm(vector) > 0:
            interests.append(vector)
    return {
        "text": text,
        "entity": entity,
        "recent_text": recent,
        "interests": np.asarray(interests, np.float32).reshape(-1, bundle.cfg.text_dim),
        "history_rows": history,
    }

In [ ]:
def saved_stage_ready(root, settings, stage):
    """Reuse disk outputs only when settings, inputs and output hashes agree."""
    path = Path(root) / "3_Representations/artifacts" / f"{stage}_manifest.json"
    if not path.exists():
        return False
    meta = json.loads(path.read_text())
    key = "config_hash" if stage == "profiles" else "representation_config_hash"
    expected = (
        settings.fingerprint()
        if stage == "profiles"
        else settings.representation_fingerprint()
    )
    if meta[key] != expected:
        raise ValueError(
            "Saved settings differ. Set REBUILD=True and rerun dependent notebooks."
        )
    for kind in ("sources", "outputs"):
        for relative, expected_hash in meta[kind].items():
            file = Path(root) / relative
            if not file.exists() or sha256(file) != expected_hash:
                raise ValueError(
                    f"Saved {stage} artifacts are stale or incomplete: {relative}. Set REBUILD=True."
                )
    return True

## Build or load saved outputs

## 1. Inspect the model contract

Core groups: content, history, match, repeated exposure, smoothed popularity and observed age.
Report extensions: three-interest matching and supported past-only half-life. Request context is provided
for ablation; slate relative advantage is opt-in (`context_features=true`) and uses the complete original
logged slate before sampling. At inference it must use the actual retrieved candidate set.

IDs, targets, shuffled position, slate length and current click-count summaries are outside the model allowlist.
Missing cosines/half-life use zero plus masks. Half-life compares disjoint completed six-hour windows,
uses Beta(1,1) smoothed CTR, and requires configurable support (100 exposures per window by default).
This specifies a new reproducible estimator; the report's original notebook is absent.

In [ ]:
schema = feature_schema(SETTINGS)
display(pd.DataFrame(schema["features"]))
assert not set(schema["model_columns"]) & set(schema["excluded"])

## 2. Replay and write features

Prequential mode freezes fitted representation/model parameters but updates observed feedback after each
timestamp. Frozen mode consumes earlier splits only and needs `TARGET_SPLIT`. Both modes advance the clock;
age and time-window queries still use the request time. The shared train/validation boundary second is processed
as one batch. `OVERWRITE=True` rebuilds only the selected run directory.

Training sampling is a seeded per-positive draw of up to four exposed negatives without replacement.
The pointwise feature table deduplicates the union of selected pairs; grouped models use the separate
positive-to-negative membership table. All popularity state is calculated before this sampling.

The SQLite user/article history bounds RAM; it requires local disk space. Full-run time, disk size and peak
RAM on MIND-large must be measured: a successful smoke run is not a full-scale performance claim.

In [ ]:
# Notebook default is a small chronological smoke build. None means full data.
MAX_TIMESTAMPS = (
    None
    if os.environ.get("NEWSREC_FULL_FEATURES") == "1"
    else int(os.environ.get("NEWSREC_SMOKE_TIMESTAMPS", "1000"))
)
TARGET_SPLIT = None  # use validation/test with update_mode='frozen'
OVERWRITE = REBUILD or os.environ.get("NEWSREC_OVERWRITE_FEATURES") == "1"
run_name = f"{SETTINGS.update_mode}-{TARGET_SPLIT or 'all'}"
if MAX_TIMESTAMPS is not None:
    run_name += f"-smoke-{MAX_TIMESTAMPS}"
run_dir = ROOT / "4_Feature_Engineering/artifacts" / run_name
if run_dir.exists() and not OVERWRITE:
    cached_run = open_feature_run(ROOT, run_dir, allow_smoke=MAX_TIMESTAMPS is not None)
    if cached_run["manifest"]["config_hash"] != SETTINGS.fingerprint():
        raise ValueError("Saved feature settings differ. Set REBUILD=True.")
    print("Loaded saved feature run:", run_dir)
else:
    run_dir = build_ranking_dataset(
        ROOT,
        SETTINGS,
        target_split=TARGET_SPLIT,
        max_timestamps=MAX_TIMESTAMPS,
        overwrite=OVERWRITE,
    )
print(run_dir)
display(json.loads((run_dir / "run_manifest.json").read_text())["rows_written"])

## 3. Audit coverage, sampling and values

Reports describe all scored logged pairs before training sampling; target tables are stored separately.
Do not load the full MIND feature dataset into pandas at once: select columns or use Arrow/Polars scans.
A smoke starting at training start may contain no validation/test output; absent partitions are expected.

In [ ]:
display(pd.read_csv(run_dir / "coverage.csv"))
display(pd.read_csv(run_dir / "feature_quality.csv").head(20))
display(pd.read_csv(run_dir / "feature_cost.csv"))
display(json.loads((run_dir / "sampling_summary.json").read_text()))
import pyarrow.dataset as ds

training_parts = list((run_dir / "ranking_features_train").glob("*.parquet"))
if training_parts:
    first_batch = next(
        ds.dataset(training_parts, format="parquet").scanner(batch_size=8).to_batches()
    )
    display(first_batch.to_pandas())
    assert np.isfinite(
        first_batch.to_pandas()[schema["model_columns"]].to_numpy()
    ).all()

## 4. Verify unlabelled inference parity

The later retrieval notebook supplies `(impression_uid, news_code)` pairs from the eligible catalogue.
No label/position is required. This initial request example uses the same pairs as the logged slate to
verify identical features. For a different Top-M set, only optional candidate-set context may change.

In [ ]:
bundle = Bundle(ROOT, SETTINGS)
state = TemporalState(bundle, *time_horizon(ROOT), SETTINGS)
try:
    t, requests, logged = next(replay_batches(ROOT, SETTINGS))
    state.begin(requests, t)
    a = build_features(requests, logged, state, bundle, SETTINGS, "logged")
    b = build_features(
        requests,
        logged[["impression_uid", "news_code"]],
        state,
        bundle,
        SETTINGS,
        "retrieved",
    )
    pd.testing.assert_frame_equal(
        a[schema["model_columns"]], b[schema["model_columns"]]
    )
    print("Logged/retrieved feature parity passed; no target fields needed.")
finally:
    state.close()

## Handoff and limitations

Read `feature_schema.json` and `run_manifest.json` before loading any partitions. Use only `model_columns`
and apply train-fitted downstream preprocessing. Each run contains keyed labels, request profile references,
negative groups, sampling summary, coverage and quality audits. A run must have `run_status.json: complete`;
its manifest explicitly identifies smoke/full status and temporal mode. Unshown retrieved items remain unlabelled.
Graph embeddings, K-means, learned contextual encoders, ANN retrieval and model training belong to later work.
Core feature/runtime coverage does not establish ranking improvement or live CTR uplift.